# Telco Customer Churn — Predictive Modeling (Step 12-18)

Notebook นี้ต่อจาก **`telco_churn_eda.ipynb`** (Step 1-11: Data Cleaning + EDA) เพื่อสร้างโมเดล Machine Learning ที่ทำนาย Churn
และเปรียบเทียบ feature importance กับ insight เชิงสถิติที่เจอใน EDA

**ก่อนรัน notebook นี้:** ต้องรัน `telco_churn_eda.ipynb` ให้จบก่อนอย่างน้อย 1 ครั้ง เพื่อสร้างไฟล์ `data/telco_churn_clean.csv`
ที่ notebook นี้จะโหลดมาใช้ต่อ


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Tahoma'  # รองรับข้อความภาษาไทยในกราฟ (เหตุผลเดียวกับที่อธิบายไว้ใน Step 3 ของ EDA notebook)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# โหลดข้อมูลที่ clean แล้วจาก telco_churn_eda.ipynb กลับมา (index_col='customerID' เพราะตอนเซฟไว้ ใช้ customerID เป็น index)
df = pd.read_csv('../data/processed/telco_churn_clean.csv', index_col='customerID')

print(f"โหลดข้อมูลสำเร็จ: {df.shape[0]} แถว x {df.shape[1]} คอลัมน์")
df.head()

โหลดข้อมูลสำเร็จ: 7032 แถว x 21 คอลัมน์


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket
customerID,,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-12
5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-48
3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-12
7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48
9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-12


**สิ่งที่โหลดมาได้:** dataframe ที่ผ่านการ clean แล้วจาก EDA notebook ครบทุกจุด (`TotalCharges` เป็นตัวเลข, ไม่มี 11 แถวที่
tenure=0, `SeniorCitizen` เป็น Yes/No) รวมถึงคอลัมน์ `tenure_bucket` ที่สร้างไว้ใน Step 8 (จะไม่ถูกใช้ในโมเดล เพราะซ้ำซ้อนกับ `tenure`)

ต่อไปนี้คือ Step 12 ที่ย้ายมาจาก `telco_churn_eda.ipynb` (feature engineering & encoding) แล้วต่อด้วย Step 13-18


In [2]:
# กำหนดรายชื่อคอลัมน์ categorical และ numeric ใหม่ (เหมือนกับที่นิยามไว้ใน Step 4 ของ EDA notebook)
# จำเป็นต้องนิยามใหม่ในไฟล์นี้ เพราะตัวแปรในหน่วยความจำ (memory) ของ notebook คนละไฟล์ไม่เชื่อมถึงกัน มีแค่ไฟล์ CSV ที่ส่งต่อข้อมูลจริงๆ เท่านั้น
categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                     'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                     'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# ส่วนที่ 2: Predictive Modeling (Step 12-18)

Step 1-11 ที่ผ่านมาเป็นการสำรวจข้อมูล (EDA) ล้วนๆ ตอนนี้จะเริ่มสร้าง **โมเดล Machine Learning** เพื่อทำนายว่าลูกค้าคนไหนจะ churn
และดูว่าปัจจัยไหนมีน้ำหนักมากที่สุดในการทำนาย (feature importance) เทียบกับ insight เชิงสถิติที่เจอใน EDA ว่าตรงกันหรือไม่

## Step 12: Feature Engineering & Encoding for ML (เตรียมข้อมูลให้พร้อมสำหรับโมเดล)

**จะทำอะไร:** โมเดล Machine Learning ทุกชนิดรับ input เป็น **ตัวเลขเท่านั้น** ไม่สามารถเข้าใจข้อความอย่าง "Yes"/"No" หรือ "Month-to-month"
ได้โดยตรง Step นี้จึงต้องแปลงข้อมูลทั้งหมดให้เป็นตัวเลข แล้วแบ่งเป็นชุด train/test ก่อนเริ่มเทรนโมเดลใน Step 14-15 แบ่งเป็น 4 ขั้นตอนย่อย:
(12.1) เข้ารหัส target (12.2) เข้ารหัส categorical features (12.3) แบ่ง train/test (12.4) ปรับสเกลตัวเลข


### 12.1 เข้ารหัส Target Variable

**จะทำอะไร:** แปลงคอลัมน์ `Churn` จากข้อความ "Yes"/"No" ให้เป็นตัวเลข 1/0 (1 = churn, 0 = ไม่ churn)

**ทำไมต้องทำ:** โมเดล classification ของ scikit-learn ต้องการให้ target เป็นตัวเลข (หรืออย่างน้อยต้องเข้ารหัสภายในเป็นตัวเลขอยู่ดี)
การแปลงเองไว้ล่วงหน้าทำให้ควบคุมได้ชัดเจนว่า 1 หมายถึงอะไร (สะดวกตอนอ่านผลลัพธ์ทีหลัง เช่น ความน่าจะเป็นที่โมเดลทำนายจะหมายถึง "โอกาส churn")


In [3]:
# (df['Churn'] == 'Yes') สร้าง Series ของค่า True/False ทีละแถว ตามว่าแถวนั้น Churn เป็น Yes หรือไม่
# .astype(int) แปลง True/False ให้เป็นตัวเลข 1/0 (True กลายเป็น 1, False กลายเป็น 0)
df['Churn_numeric'] = (df['Churn'] == 'Yes').astype(int)

df[['Churn', 'Churn_numeric']].head()

,Churn,Churn_numeric
customerID,,
7590-VHVEG,No,0
5575-GNVDE,No,0
3668-QPYBK,Yes,1
7795-CFOCW,No,0
9237-HQITU,Yes,1


**ผลลัพธ์หมายความว่าอย่างไร:** ตอนนี้มีคอลัมน์ `Churn_numeric` เพิ่มขึ้นมา ซึ่งเป็นตัวเลข 1/0 ที่มีความหมายตรงกับ `Churn` เดิมทุกประการ
(Yes → 1, No → 0) พร้อมนำไปใช้เป็น target ของโมเดลในขั้นตอนถัดไป


### 12.2 One-Hot Encoding สำหรับ Categorical Features

**จะทำอะไร:** แปลงตัวแปรหมวดหมู่ทั้ง 16 คอลัมน์ (ตัวแปรเดียวกับที่ใช้ใน Step 4-5) ให้เป็นตัวเลขด้วยเทคนิคที่เรียกว่า **One-Hot Encoding**

**One-Hot Encoding คืออะไร (แบบเข้าใจง่าย):** สมมติคอลัมน์ `Contract` มี 3 ค่า (Month-to-month, One year, Two year) One-Hot Encoding
จะสร้างคอลัมน์ใหม่แยกกัน 1 คอลัมน์ต่อ 1 ค่า เช่น `Contract_One year` (มีค่า 1 ถ้าลูกค้าคนนั้นทำสัญญา 1 ปี, มีค่า 0 ถ้าไม่ใช่) และ
`Contract_Two year` (เหมือนกัน) — เปรียบเหมือนตั้งคำถามใช่/ไม่ใช่แยกกันสำหรับแต่ละตัวเลือก แทนที่จะเก็บเป็นข้อความเดียว

**ทำไมต้องใช้ `drop_first=True`:** ถ้าเรารู้ว่าลูกค้าไม่ใช่ Month-to-month และไม่ใช่ One year ก็สรุปได้เองว่าต้องเป็น Two year แน่นอน
(ไม่ต้องมีคอลัมน์ `Contract_Two year` ก็รู้คำตอบได้) การเก็บครบทุกค่าจะทำให้เกิดปัญหาที่เรียกว่า **dummy variable trap** — คอลัมน์ต่างๆ
ทำนายกันเองได้แบบสมบูรณ์ (perfect multicollinearity) ซึ่งจะทำให้ผลลัพธ์ของ Logistic Regression ใน Step 14 ตีความยากขึ้น การใส่
`drop_first=True` จะตัดค่าตัวแรกออกไป 1 ค่าเสมอ (เหลือไว้เป็นค่า "ฐาน" หรือ baseline ให้เทียบ) แก้ปัญหานี้ได้


In [4]:
# pd.get_dummies() คือฟังก์ชันสำเร็จรูปของ pandas สำหรับทำ One-Hot Encoding
# ใส่เฉพาะคอลัมน์ categorical_cols (16 ตัวจาก Step 4) ไม่รวมตัวเลขหรือ target
# drop_first=True ตัดค่าตัวแรกของแต่ละคอลัมน์ออก 1 ค่าเสมอ ตามเหตุผลที่อธิบายไว้ข้างบน
X_categorical = pd.get_dummies(df[categorical_cols], drop_first=True)

print(f"จำนวนคอลัมน์ก่อนเข้ารหัส: {len(categorical_cols)}")
print(f"จำนวนคอลัมน์หลังเข้ารหัส (One-Hot): {X_categorical.shape[1]}")
X_categorical.head()

จำนวนคอลัมน์ก่อนเข้ารหัส: 16
จำนวนคอลัมน์หลังเข้ารหัส (One-Hot): 27


,gender_Male,SeniorCitizen_Yes,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
customerID,,,,,,,,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False
5575-GNVDE,True,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True
3668-QPYBK,True,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
7795-CFOCW,True,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False
9237-HQITU,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False


**ผลลัพธ์หมายความว่าอย่างไร:** จาก 16 คอลัมน์เดิม กลายเป็นคอลัมน์ตัวเลขจำนวนมากขึ้น (เพราะบางคอลัมน์เช่น `InternetService`,
`PaymentMethod` มีมากกว่า 2 ค่า จึงถูกแตกเป็นหลายคอลัมน์) แต่ละคอลัมน์ใหม่มีค่าแค่ 0 หรือ 1 เท่านั้น (คอลัมน์แบบนี้เรียกว่า dummy variable)
พร้อมนำไปใช้เป็น input ของโมเดลได้แล้ว


In [5]:
# รวมคอลัมน์ตัวเลขเดิม (numeric_cols จาก Step 4) เข้ากับคอลัมน์ dummy ที่เพิ่งสร้าง ให้เป็นตาราง feature เดียว (X)
# pd.concat([...], axis=1) แปลว่า "เอาตารางมาต่อกันตามแนวคอลัมน์" (axis=1 คือต่อข้าง ต่างจาก axis=0 ที่ต่อแถว)
X = pd.concat([df[numeric_cols], X_categorical], axis=1)

# y คือ target ที่จะให้โมเดลทำนาย (ค่าที่เพิ่งเข้ารหัสไว้ใน 12.1)
y = df['Churn_numeric']

print(f"Feature matrix X: {X.shape[0]} แถว x {X.shape[1]} คอลัมน์")
print(f"Target y: {y.shape[0]} แถว")

Feature matrix X: 7032 แถว x 30 คอลัมน์
Target y: 7032 แถว


**หมายเหตุ:** ตัวแปร `tenure_bucket` ที่สร้างไว้ใน Step 8 (สำหรับทำตารางไขว้) **ไม่ถูกนำมาใส่ใน X** เพราะมันเป็นแค่การแบ่งช่วงของ
`tenure` ที่มีอยู่แล้ว การใส่ทั้งสองตัวพร้อมกันจะทำให้เกิดข้อมูลซ้ำซ้อน (redundant) โดยไม่ได้ประโยชน์เพิ่ม


### 12.3 แบ่งข้อมูลเป็น Train/Test Set

**จะทำอะไร:** แบ่งลูกค้าทั้งหมดออกเป็น 2 กลุ่มแบบสุ่ม — **80% สำหรับ "สอน" โมเดล (training set)** และ **20% สำหรับ "สอบ" โมเดล
(test set)** โดยที่โมเดลจะไม่เคยเห็นข้อมูลใน test set เลยระหว่างเทรน

**ทำไมต้องแบ่ง:** ถ้าเอาข้อมูลเดียวกันมาทั้งสอนและสอบ โมเดลอาจแค่ "จำ" คำตอบได้หมดโดยไม่ได้เรียนรู้ pattern ที่ใช้ได้จริง (ปัญหานี้เรียกว่า
**overfitting**) การแบ่งข้อมูลที่โมเดลไม่เคยเห็นมาทดสอบ จะบอกได้ว่าโมเดลเก่งจริงหรือแค่จำข้อสอบมา

**ทำไมต้องใช้ `stratify=y`:** จาก Step 3 เรารู้ว่าข้อมูลไม่สมดุล (churn แค่ 26.6%) ถ้าแบ่งแบบสุ่มธรรมดา อาจบังเอิญได้ train set ที่มี
สัดส่วน churn ต่างจาก test set มาก (เช่น train มี churn 30% แต่ test มีแค่ 20%) ทำให้เปรียบเทียบผลลัพธ์ได้ไม่แม่นยำ `stratify=y`
บังคับให้สัดส่วน churn ใน train และ test ใกล้เคียงกับสัดส่วนจริง (26.6%) ทั้งคู่


In [6]:
from sklearn.model_selection import train_test_split

# test_size=0.2 แปลว่าแบ่งไป test set 20% (เหลือ train 80%)
# stratify=y บังคับให้สัดส่วนของ y (churn/ไม่ churn) เท่ากันทั้งใน train และ test
# random_state=42 ล็อกการสุ่มให้ได้ผลลัพธ์เดิมทุกครั้งที่รัน (reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train set: {X_train.shape[0]} แถว ({y_train.mean() * 100:.1f}% churn)")
print(f"Test set: {X_test.shape[0]} แถว ({y_test.mean() * 100:.1f}% churn)")

Train set: 5625 แถว (26.6% churn)
Test set: 1407 แถว (26.6% churn)


**ผลลัพธ์หมายความว่าอย่างไร:** ยืนยันว่า `stratify=y` ทำงานถูกต้อง — สัดส่วน churn ใน train set และ test set ใกล้เคียงกับ
base rate 26.6% ทั้งคู่ (ต่างกันเพียงเศษเสี้ยว % เพราะการปัดเศษจากการแบ่งเป็นจำนวนเต็มคน) พร้อมนำไปเทรนโมเดลใน Step 14-15


### 12.4 ปรับสเกลตัวเลข (Feature Scaling)

**จะทำอะไร:** ปรับค่าตัวเลข 3 คอลัมน์ (`tenure`, `MonthlyCharges`, `TotalCharges`) ให้อยู่ใน**สเกลเดียวกัน** โดยใช้เทคนิคที่ชื่อ
**StandardScaler** ซึ่งจะแปลงแต่ละค่าให้มีค่าเฉลี่ย (mean) เป็น 0 และส่วนเบี่ยงเบนมาตรฐาน (std) เป็น 1

**ทำไมต้อง scale:** สังเกตจาก Step 4 ว่า `tenure` มีค่าตั้งแต่ 1-72 ในขณะที่ `TotalCharges` มีค่าตั้งแต่ ~19 ถึง ~8,700 — ถ้าไม่ปรับสเกล
โมเดลอย่าง Logistic Regression (Step 14) ที่คำนวณระยะห่าง/น้ำหนักตามขนาดตัวเลข จะให้ "น้ำหนัก" กับ `TotalCharges` มากเกินจริงเพียงเพราะ
ตัวเลขมันใหญ่กว่า ทั้งที่ไม่ได้เกี่ยวข้องกับ Churn มากกว่าจริง การ scale ให้ทุกคอลัมน์อยู่ในขนาดใกล้เคียงกันจะทำให้เปรียบเทียบน้ำหนัก
ความสำคัญของแต่ละตัวแปรได้อย่างเป็นธรรม (Random Forest ใน Step 15 ไม่จำเป็นต้อง scale แต่ scale ไว้ก็ไม่ทำให้ผลลัพธ์แย่ลง)

**ทำไมต้อง fit บน train set เท่านั้น:** `StandardScaler` ต้องคำนวณค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานมาก่อนถึงจะปรับสเกลได้ ถ้าคำนวณค่าเฉลี่ย
จากข้อมูลทั้งหมด (รวม test set ด้วย) จะเท่ากับให้โมเดล "แอบเห็น" ข้อมูลของ test set ตั้งแต่ก่อนเทรน (เรียกว่า **data leakage**)
ซึ่งจะทำให้ผลการทดสอบดูดีเกินจริงกว่าที่ควรจะเป็นจริงเมื่อเอาไปใช้กับข้อมูลใหม่ในโลกจริง


In [7]:
from sklearn.preprocessing import StandardScaler

# .copy() ป้องกัน warning ของ pandas เวลาจะแก้ไขค่าในตารางที่ตัดมาจากตารางใหญ่กว่า (X)
X_train = X_train.copy()
X_test = X_test.copy()

scaler = StandardScaler()

# .fit_transform() บน train: คำนวณค่าเฉลี่ย/std จาก train set (fit) แล้วปรับสเกลค่าของ train set เลยในขั้นตอนเดียว (transform)
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# .transform() บน test: ใช้ค่าเฉลี่ย/std ที่ "จำ" ไว้จาก train set มาปรับสเกล test set (ไม่คำนวณค่าเฉลี่ยใหม่จาก test)
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("ค่าเฉลี่ยของตัวเลขใน train set หลัง scale (ควรใกล้ 0 มาก):")
print(X_train[numeric_cols].mean().round(3))

ค่าเฉลี่ยของตัวเลขใน train set หลัง scale (ควรใกล้ 0 มาก):
tenure           -0.0
MonthlyCharges    0.0
TotalCharges      0.0
dtype: float64


**ผลลัพธ์หมายความว่าอย่างไร:** ค่าเฉลี่ยของทั้ง 3 คอลัมน์ใน train set หลัง scale ใกล้เคียง 0 มาก (เป็นไปตามหลักการของ
StandardScaler) ยืนยันว่าการปรับสเกลทำงานถูกต้อง ส่วน test set จะไม่ได้มีค่าเฉลี่ยเป๊ะ 0 (เพราะ scale ด้วยค่าเฉลี่ยของ train ไม่ใช่ของตัวเอง)
ซึ่งเป็นเรื่องปกติและเป็นสิ่งที่ต้องการ — ป้องกัน data leakage ตามที่อธิบายไว้ข้างบน


**สรุป Step 12:** เตรียมข้อมูลพร้อมสำหรับโมเดลแล้วครบทุกขั้นตอน — เข้ารหัส target เป็น 0/1, แปลง categorical เป็น dummy variables
ด้วย One-Hot Encoding, แบ่ง train (80%) / test (20%) แบบรักษาสัดส่วน churn ไว้เท่ากัน, และปรับสเกลตัวเลขโดยไม่ให้เกิด data leakage

ขั้นตอนถัดไป (Step 13) จะแก้ปัญหาข้อมูล imbalanced ด้วย SMOTE ก่อนนำไปเทรนโมเดลจริงใน Step 14-15
